# Data pipeline: DebateGPT primary, args.me secondary

This notebook downloads or loads public DebateGPT data, filters the human-human condition, validates the four agreement fields, and prepares args.me. It never creates replacement rows.

In [1]:
from pathlib import Path
import json
import os
import sys

root = Path.cwd()
while root != root.parent and not (root / 'src').is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from src import data_loader as dl


In [2]:
debategpt_path = root / 'data' / 'raw' / 'debategpt'
debategpt_files = [p for p in debategpt_path.glob('*') if p.is_file() and p.suffix.lower() in {'.json', '.jsonl', '.csv'}]
if not debategpt_files:
    import subprocess
    subprocess.run([sys.executable, str(root / 'scripts' / 'download_required_datasets.py')], check=True)
    print({'status': 'downloaded', 'path': str(debategpt_path)})
else:
    print({'status': 'using_local_export', 'path': str(debategpt_path)})


{'status': 'using_local_export', 'path': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/raw/debategpt'}


In [3]:
debategpt = dl.load_debategpt(data_path=str(debategpt_path), checkpoint_dir=str(root / 'checkpoints'), force=True)
args_path = root / 'data' / 'raw' / 'args_me_corpus'
args_me = dl.load_args_me_corpus(data_path=str(args_path), checkpoint_dir=str(root / 'checkpoints'), force=False)
instances = dl.build_stance_change_instances(debategpt)
report = {
    'primary_dataset': 'DebateGPT',
    'condition': 'human-human',
    'debategpt_path': str(debategpt_path),
    'debategpt_rows': int(len(debategpt)),
    'debategpt_columns': list(debategpt.columns),
    'movement_rows': int(len(instances)),
    'movement_classes': instances['movement_class'].value_counts().to_dict(),
    'args_me_rows': int(len(args_me)),
    'sample': debategpt.head(5).to_dict(orient='records'),
}
report_path = root / 'reports' / 'data_verification.json'
report_path.write_text(json.dumps(report, indent=2, default=str) + '\n', encoding='utf-8')
print({'status': 'verified', 'report': str(report_path), 'debategpt_rows': len(debategpt), 'args_me_rows': len(args_me)})


[09:38:25] INFO belief_debate_analyzer: Checkpoint miss: debategpt_v2__efa5031df430.pkl (computing)
[09:38:25] INFO belief_debate_analyzer: Loaded real DebateGPT rows from /home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/raw/debategpt (300 rows)
[09:38:25] INFO belief_debate_analyzer: Checkpoint hit: args_me__6543d0e2e601.pkl (skipping recompute)


{'status': 'verified', 'report': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/reports/data_verification.json', 'debategpt_rows': 300, 'args_me_rows': 400}


## Prepare the human-annotation sample (paper §5.2)

H1/H2 require three independent human annotators to label real stance-change instances with a primary cause (evidence adoption, anchoring, echo, or strategic persuasion). This step produces only the reproducible *sample* and the *guidelines* an annotator needs -- the exact sampling procedure the paper commits to reporting. It cannot and does not produce `data/annotations/human_labels.csv` itself: those labels must come from three real people reading real transcripts, and fabricating them would violate this repository's non-negotiable rules.

**Annotation guidelines** (verbatim category definitions from the paper, §3.3): for the highlighted participant's stance change in each instance below, choose the single primary cause:

- **Evidence adoption**: the turn contains a new evidence snippet (empirical or causal) with high argument quality, semantically dissimilar to the participant's own prior turns.
- **Anchoring**: the participant's utterance is semantically highly similar to its own prior stance expression, or the stance change is small despite high-quality counter-evidence.
- **Echo / peer influence**: the participant's utterance has high semantic similarity to the other side's immediately preceding turn, or adopts the other side's rhetorical strategy distribution.
- **Strategic persuasion**: the other side's rhetorical strategy shifted abruptly (e.g. causal/empirical to emotional/moral) and the participant's stance changed substantially in response.

If none clearly applies, or the instance is genuinely ambiguous, record a tie/ambiguous note rather than forcing a label (paper §4.2 procedure).


In [7]:
movement_instances = dl.build_stance_change_instances(debategpt)
sample = dl.sample_for_annotation(movement_instances, n=150, seed=42)

transcripts_by_id = {t['transcript_id']: t for t in dl.build_transcripts(debategpt)}


def _render_transcript(group_id):
    turns = transcripts_by_id[str(group_id)]['turns']
    return "\n".join(f"[{turn['speaker']}] {turn['text']}" for turn in turns)


sample_out = sample[[
    'instance_id', 'group_id', 'participant_id', 'topic', 'movement_class',
    'agreementPreTreatment', 'agreementPostTreatment',
    'sideAgreementPreTreatment', 'sideAgreementPostTreatment',
]].copy()
sample_out['transcript_text'] = sample_out['group_id'].map(_render_transcript)

annotations_dir = root / 'data' / 'annotations'
annotations_dir.mkdir(parents=True, exist_ok=True)

# researcher-only master copy, kept separate from what annotators see: movement_class
# must not be shown to annotators, since revealing our own joint/divergent split would
# bias exactly the H2 comparison that split is meant to test independently.
master_path = annotations_dir / 'annotation_sample_master.csv'
sample_out.to_csv(master_path, index=False)

annotator_cols = [
    'instance_id', 'group_id', 'participant_id', 'topic',
    'agreementPreTreatment', 'agreementPostTreatment',
    'sideAgreementPreTreatment', 'sideAgreementPostTreatment',
    'transcript_text',
]
ANNOTATOR_IDS = ['TS', 'GS', 'RK']

INSTRUCTIONS = f"""Belief-Tracking Debate Analyzer -- human annotation task
Assigned annotator ID: {{annotator_id}}

TASK: For each of the {len(sample_out)} rows below, read the full transcript in
`transcript_text` and focus on the participant named in `participant_id` (their
lines are tagged [participant_id] in the transcript). Decide the SINGLE primary
cause of that participant's stance change across the debate, and write exactly
one of the following five values into the `label` column for that row:

  evidence_adoption    - a new empirical/causal argument, high quality, and
                         unlike anything the participant said earlier, changed
                         their mind.
  anchoring            - the participant mostly repeated their own prior
                         position, or barely moved despite strong counter-
                         evidence.
  echo                 - the participant's later turns closely mirror the
                         other side's wording, framing, or rhetorical style.
  strategic_persuasion - the other side abruptly shifted from causal/empirical
                         to emotional/moral appeals, and the participant's
                         stance moved substantially right after.
  ambiguous             - none of the above clearly fits, or more than one
                         seems equally plausible. Do not force a choice.

RULES:
  - Work independently. Do not discuss instances or compare answers with the
    other two annotators before all three of you have finished.
  - One label per row, in the `label` column, using one of the five values above
    exactly as spelled (lowercase, underscores).
  - Leave every other column unchanged.
  - This file is your personal copy (annotator ID {{annotator_id}}); do not
    merge or overwrite the other annotators' files.
"""

annotator_frames = {}
for annotator_id in ANNOTATOR_IDS:
    # independent per-annotator shuffle (instance_id preserved) guards against
    # position/fatigue bias; the master file above stays in the original order.
    seed = 42 + ANNOTATOR_IDS.index(annotator_id) + 1
    annotator_df = sample_out[annotator_cols].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    annotator_df.insert(0, 'annotator', annotator_id)
    annotator_df['label'] = ''
    annotator_frames[annotator_id] = annotator_df
    annotator_path = annotations_dir / f'annotation_sample_{annotator_id}.csv'
    with open(annotator_path, 'w', encoding='utf-8', newline='') as handle:
        for line in INSTRUCTIONS.format(annotator_id=annotator_id).splitlines():
            handle.write(f"# {line}\n")
        handle.write("#\n")
        annotator_df.to_csv(handle, index=False)

print({
    'status': 'annotation_sample_ready',
    'n_instances': len(sample_out),
    'movement_class_counts': sample_out['movement_class'].value_counts().to_dict(),
    'master_path': str(master_path),
    'annotator_csv_paths': [str(annotations_dir / f'annotation_sample_{a}.csv') for a in ANNOTATOR_IDS],
    'next_step': (
        'Run the next cell to also generate a print-ready PDF packet per annotator. '
        'Send each annotator their own CSV or PDF only. Once all three return their '
        'labels, concatenate into long format (instance_id, annotator, label) as '
        'data/annotations/human_labels.csv before H1/H2 can run.'
    ),
})




{'status': 'annotation_sample_ready', 'n_instances': 145, 'movement_class_counts': {'joint_movement': 76, 'divergent_movement': 69}, 'master_path': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/annotations/annotation_sample_master.csv', 'annotator_csv_paths': ['/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/annotations/annotation_sample_TS.csv', '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/annotations/annotation_sample_GS.csv', '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/annotations/annotation_sample_RK.csv'], 'next_step': 'Run the next cell to also generate a print-ready PDF packet per annotator. Send each annotator their own CSV or PDF only. Once all three return their labels, concatenate into long format (instance_id, annotator, label) as data/annotations/human_labels.csv before H1/H2 can run.'}


In [9]:
from fpdf import FPDF

CATEGORY_DEFS = [
    ('evidence_adoption', 'A new empirical/causal argument, high quality and unlike anything '
                          'the participant said earlier, changed their mind.'),
    ('anchoring', 'The participant mostly repeated their own prior position, or barely moved '
                  'despite strong counter-evidence.'),
    ('echo', "The participant's later turns closely mirror the other side's wording, framing, "
             'or rhetorical style.'),
    ('strategic_persuasion', 'The other side abruptly shifted from causal/empirical to '
                             "emotional/moral appeals, and the participant's stance moved "
                             'substantially right after.'),
    ('ambiguous', 'None of the above clearly fits, or more than one seems equally plausible. '
                  'Do not force a choice.'),
]


def _safe(text) -> str:
    # core PDF fonts are latin-1 only; replace anything outside that range rather
    # than crash on a stray curly quote or em-dash somewhere in 145 real transcripts.
    return str(text).encode('latin-1', 'replace').decode('latin-1')


def _mc(pdf, h, text):
    # fpdf2's multi_cell defaults to new_x=RIGHT, which leaves the cursor at the right
    # margin instead of the left -- explicitly reset it so consecutive calls don't run
    # out of horizontal space on the very next line.
    pdf.multi_cell(0, h, _safe(text), new_x='LMARGIN', new_y='NEXT')


def _build_annotator_pdf(annotator_id: str, ordered_df, path) -> None:
    pdf = FPDF(format='A4', unit='mm')
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.set_margins(18, 18, 18)

    pdf.add_page()
    pdf.set_font('Helvetica', 'B', 16)
    _mc(pdf, 8, 'Belief-Tracking Debate Analyzer -- Human Annotation Task')
    pdf.set_font('Helvetica', '', 11)
    pdf.ln(2)
    _mc(pdf, 6, f'Assigned annotator ID: {annotator_id}')
    _mc(pdf, 6, f'Total instances in this packet: {len(ordered_df)}')
    pdf.ln(3)

    pdf.set_font('Helvetica', 'B', 12)
    _mc(pdf, 6, 'Task')
    pdf.set_font('Helvetica', '', 10)
    _mc(pdf, 5,
        'For each instance below, read the full transcript and focus on the participant '
        'named under "Focus on participant" (their lines are tagged with that name). Decide '
        "the SINGLE primary cause of that participant's stance change across the debate, and "
        'check ONE of the five categories.'
    )
    pdf.ln(3)

    pdf.set_font('Helvetica', 'B', 12)
    _mc(pdf, 6, 'Categories')
    for name, definition in CATEGORY_DEFS:
        pdf.set_font('Helvetica', 'B', 10)
        _mc(pdf, 5, f'- {name}')
        pdf.set_font('Helvetica', '', 10)
        _mc(pdf, 5, f'  {definition}')
    pdf.ln(3)

    pdf.set_font('Helvetica', 'B', 12)
    _mc(pdf, 6, 'Rules')
    pdf.set_font('Helvetica', '', 10)
    for rule in [
        'Work independently. Do not discuss instances or compare answers with the other '
        'two annotators before all three of you have finished.',
        'Check exactly one category per instance; use "ambiguous" rather than forcing a '
        'choice you are not confident in.',
        f'This packet is your personal copy (annotator ID {annotator_id}); please do not '
        'share it with the other annotators.',
    ]:
        _mc(pdf, 5, f'- {rule}')

    for position, row in enumerate(ordered_df.itertuples(index=False), start=1):
        pdf.add_page()
        pdf.set_font('Helvetica', 'B', 13)
        _mc(pdf, 7, f'Instance {position} of {len(ordered_df)}   (ID: {row.instance_id})')
        pdf.set_font('Helvetica', '', 10)
        _mc(pdf, 5, f'Topic: {row.topic}')
        pdf.set_font('Helvetica', 'B', 10)
        _mc(pdf, 5, f'Focus on participant: {row.participant_id}')
        pdf.ln(2)

        pdf.set_font('Helvetica', 'B', 10)
        _mc(pdf, 5, 'Transcript:')
        pdf.set_font('Helvetica', '', 9)
        _mc(pdf, 4.5, row.transcript_text)
        pdf.ln(3)

        pdf.set_font('Helvetica', 'B', 10)
        _mc(pdf, 5, 'Your label (check ONE):')
        pdf.set_font('Helvetica', '', 10)
        for name, _ in CATEGORY_DEFS:
            _mc(pdf, 6, f'[    ]   {name}')
        pdf.ln(2)
        _mc(pdf, 5, 'Notes (optional): ______________________________________________')

    pdf.output(str(path))


annotator_pdf_paths = {}
for annotator_id, annotator_df in annotator_frames.items():
    pdf_path = annotations_dir / f'annotation_sample_{annotator_id}.pdf'
    _build_annotator_pdf(annotator_id, annotator_df, pdf_path)
    annotator_pdf_paths[annotator_id] = pdf_path

print({
    'status': 'annotator_pdfs_ready',
    'paths': {a: str(p) for a, p in annotator_pdf_paths.items()},
})


{'status': 'annotator_pdfs_ready', 'paths': {'TS': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/annotations/annotation_sample_TS.pdf', 'GS': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/annotations/annotation_sample_GS.pdf', 'RK': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/data/annotations/annotation_sample_RK.pdf'}}
